In [82]:
%%bash

## привет! здесь будет указаны библиотки необходимые для выполнения всего кода (вроде никакие не забыл)
pip install kagglehub
pip install psycopg2-binary
pip install pyspark==3.0.3

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [ ]:
import kagglehub

## скачаем датасет с kaggle
path = kagglehub.dataset_download("justinwilcher/nashville-accident-reports-jan-2018-apl-2025")

print("Path to dataset files:", path)

In [1]:
%%bash

## проверим, что находится по пути, который был выведен в консоль
ls /home/ubuntu/.cache/kagglehub/datasets/justinwilcher/nashville-accident-reports-jan-2018-apl-2025/versions/1

Nashville Accidents Jan 2018 - Apl 2025.csv


In [2]:
%%bash

## создаём папку dataset, если её нет
mkdir -p ~/dataset

## копируем csv-файл в папку dataset
cp /home/ubuntu/.cache/kagglehub/datasets/justinwilcher/nashville-accident-reports-jan-2018-apl-2025/versions/1/*.csv ~/dataset/
mv "dataset/Nashville Accidents Jan 2018 - Apl 2025.csv" dataset/nashville_accidents_2018_2025.csv


In [3]:
%%bash

ls ~/dataset/


nashville_accidents_2018_2025.csv


In [83]:
%%bash

if hdfs dfs -test -e /user/ubuntu/nashville_accidents/nashville_accidents_2018_2025.csv; then
    hdfs dfs -rm /user/ubuntu/nashville_accidents/nashville_accidents_2018_2025.csv
fi

hdfs dfs -mkdir -p /user/ubuntu/nashville_accidents

hdfs dfs -put dataset/nashville_accidents_2018_2025.csv /user/ubuntu/nashville_accidents/


Deleted /user/ubuntu/nashville_accidents/nashville_accidents_2018_2025.csv


In [84]:
%%bash

## проверим, что файл действительно лежит в нужной директории
hdfs dfs -ls /user/ubuntu/nashville_accidents/

Found 1 items
-rw-r--r--   1 ubuntu hadoop   48744315 2025-05-01 18:32 /user/ubuntu/nashville_accidents/nashville_accidents_2018_2025.csv


In [26]:
## файл загрузили, в hdfs положили -> переходим к части Spark

In [7]:
%%html
<style>
div.output_area pre {
    white-space: pre; 
}
</style>

In [52]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ETL_Hive_Project") \
    .enableHiveSupport() \
    .getOrCreate()

df = spark.read.csv("hdfs:///user/ubuntu/nashville_accidents/nashville_accidents_2018_2025.csv", header=True, inferSchema=True)

df.printSchema()

root
 |-- Accident Number: long (nullable = true)
 |-- Date and Time: string (nullable = true)
 |-- Number of Motor Vehicles: integer (nullable = true)
 |-- Number of Injuries: integer (nullable = true)
 |-- Number of Fatalities: integer (nullable = true)
 |-- Property Damage: string (nullable = true)
 |-- Hit and Run: string (nullable = true)
 |-- Collision Type Description: string (nullable = true)
 |-- Weather Description: string (nullable = true)
 |-- Illumination Description: string (nullable = true)
 |-- Street Address: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Precinct: string (nullable = true)
 |-- Lat: double (nullable = true)
 |-- Long: double (nullable = true)
 |-- HarmfulCodes: string (nullable = true)
 |-- HarmfulDescriptions: string (nullable = true)
 |-- ObjectId: integer (nullable = true)
 |-- Zip Code: integer (nullable = true)
 |-- RPA: integer (nullable = true)
 |-- Weather: integer (nullable = true)
 |-- I

In [53]:
df.show(5,truncate=False, vertical=True)

-RECORD 0-------------------------------------------------------------
 Accident Number            | 2008473471                              
 Date and Time              | 7/14/2018 11:00:00 PM                   
 Number of Motor Vehicles   | 2                                       
 Number of Injuries         | 2                                       
 Number of Fatalities       | 0                                       
 Property Damage            | null                                    
 Hit and Run                | N                                       
 Collision Type Description | ANGLE                                   
 Weather Description        | NO ADVERSE CONDITIONS                   
 Illumination Description   | null                                    
 Street Address             | 28TH AVE N & JEFFERSON ST               
 City                       | NASHVILLE                               
 State                      | TN                                      
 Preci

In [54]:
import pyspark.sql.functions as F

## преобразуем столбец Date and Time
df = df.withColumn('date', F.to_timestamp('Date and Time', 'M/d/yyyy h:mm:ss a'))

In [55]:
def create_first_table(df):
    df_filtered = df.filter(
        (F.year('date') == 2018) & (F.month('date') == 1) ## здесь устанавливаем тестовую дату (год+месяц)
    )

    first_table = df_filtered.select(
        'Accident Number',
        'date',
        'State',
        'Number of Injuries',
        'Number of Fatalities'
    )
    
    first_table = first_table \
        .withColumnRenamed('Accident Number', 'accident_number') \
        .withColumnRenamed('State', 'state') \
        .withColumnRenamed('Number of Injuries', 'number_of_injuries') \
        .withColumnRenamed('Number of Fatalities', 'number_of_fatalities')
    
    spark.sql("CREATE DATABASE IF NOT EXISTS etl_project_db")
    first_table.write.mode('append') \
        .format('orc') \
        .saveAsTable('etl_project_db.first_table')

create_first_table(df)

In [56]:
import pyspark.sql.functions as F
def create_second_table(df):
    df_filtered = df.filter(
        (F.year('date') == 2018) & (F.month('date') == 1) ## здесь устанавливаем тестовую дату (год+месяц)
    )
    
    second_table = df_filtered.select(
        'Accident Number',
        'date',
        'State',
        'City',
        'Street Address',
        'Lat',
        'Long',
        'Number of Motor Vehicles' 
    )
    
    second_table = second_table \
        .withColumnRenamed('Accident Number','accident_number') \
        .withColumnRenamed('State','state') \
        .withColumnRenamed('City','city') \
        .withColumnRenamed('Street Address','street_address') \
        .withColumnRenamed('Lat','lat') \
        .withColumnRenamed('Long','long') \
        .withColumnRenamed('Number of Motor Vehicles','number_of_motor_vehicles')
    
    spark.sql("CREATE DATABASE IF NOT EXISTS etl_project_db")
    second_table.write.mode('append') \
        .format('orc') \
        .saveAsTable('etl_project_db.second_table')
    
create_second_table(df)

In [57]:
import pyspark.sql.functions as F
def create_third_table(df):
    df_filtered = df.filter(
        (F.year('date') == 2018) & (F.month('date') == 1) ## здесь устанавливаем тестовую дату (год+месяц)
    )
    
    third_table = df_filtered.select(
        'Accident Number',
        'date',
        'Reporting Officer',
        'Property Damage',
        'Hit and Run',
        'Collision Type Description',
        'Weather Description',
        'Illumination Description'
    )
    
    third_table = third_table \
    .withColumnRenamed('Accident Number', 'accident_number') \
    .withColumnRenamed('Reporting Officer', 'reporting_officer') \
    .withColumnRenamed('Property Damage', 'property_damage') \
    .withColumnRenamed('Hit and Run', 'hit_and_run') \
    .withColumnRenamed('Collision Type Description', 'collision_type_description') \
    .withColumnRenamed('Weather Description', 'weather_description') \
    .withColumnRenamed('Illumination Description', 'illumination_description')
    
    spark.sql("CREATE DATABASE IF NOT EXISTS etl_project_db")
    third_table.write.mode('append') \
        .format('orc') \
        .saveAsTable('etl_project_db.third_table')
    
create_third_table(df)

In [3]:
## чтение first_table из Hive
first_table = spark.read.table("etl_project_db.first_table")

first_table.show(10000)

+---------------+-------------------+-----+------------------+--------------------+
|accident_number|               date|state|number_of_injuries|number_of_fatalities|
+---------------+-------------------+-----+------------------+--------------------+
|    20170000819|2018-01-01 14:45:00|   TN|                 1|                   0|
|    20170007441|2018-01-03 15:01:00|   TN|                 0|                   0|
|    20170020701|2018-01-07 13:44:00|   TN|                 2|                   0|
|    20170035976|2018-01-11 23:59:00|   TN|                 0|                   0|
|    20180000079|2018-01-01 06:26:00|   TN|                 0|                   0|
|    20180000088|2018-01-01 06:25:00|   TN|                 0|                   0|
|    20180000109|2018-01-01 06:39:00|   TN|                 0|                   0|
|    20180000167|2018-01-01 07:12:00|   TN|                 1|                   0|
|    20180000173|2018-01-01 07:00:00|   TN|                 0|              

In [1]:
## перенесём данные из Hive таблиц в Postgres таблицы
## для этого необходимо на ноде создать суперпользователя myuser (пароль: mypassword, БД: mydatabase)
## ещё необходимо положить jar-файл для взаимодействия между приложением и Postgre в корень проекта (или измените директорию)
## также укажите адрес ноды, которую вы используете для работы

node_address = "158.160.133.10"

from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder \
    .appName("ETL for PostgreSQL") \
    .config("spark.jars", "./postgresql-42.7.5.jar") \
    .getOrCreate()

In [60]:
## перенос first_table в Postgres

hive_df = spark.table("etl_project_db.first_table")

last_month_data = hive_df.filter(
    (F.year('date') == 2018) & (F.month('date') == 1) ## здесь устанавливаем тестовую дату (год+месяц) 
)

## здесь можно написать любую агрегацию (в рамках разумного)
last_month_data = last_month_data.groupBy(
    F.year('date').alias('year'),
    F.month('date').alias('month'),
    'state'
).agg(
    F.count('number_of_injuries').alias('injuries_count'),
    F.count('number_of_fatalities').alias('fatalities_count')
)

last_month_data.write \
    .format("jdbc") \
    .option("url", f"jdbc:postgresql://{node_address}:5432/mydatabase") \
    .option("dbtable", "injuries_and_fatalities_per_month") \
    .option("user", "myuser") \
    .option("password", "mypassword") \
    .mode("overwrite") \
    .save()

In [61]:
## чтение second_table из Hive
second_table = spark.read.table("etl_project_db.second_table")

second_table.show(20)

+---------------+-------------------+-----+------------+--------------------+--------+---------+------------------------+
|accident_number|               date|state|        city|      street_address|     lat|     long|number_of_motor_vehicles|
+---------------+-------------------+-----+------------+--------------------+--------+---------+------------------------+
|    20170000819|2018-01-01 14:45:00|   TN|   NASHVILLE|OLD HICKORY BLVD ...|36.27071|-86.76002|                       1|
|    20170007441|2018-01-03 15:01:00|   TN|   NASHVILLE|WEST END AV & PAR...|36.13967|-86.81874|                       2|
|    20170020701|2018-01-07 13:44:00|   TN|   NASHVILLE|I24 E EXT RAMP & ...|36.04589|-86.66055|                       1|
|    20170035976|2018-01-11 23:59:00|   TN|   NASHVILLE|LYNWOOD AV & 8TH AVS|36.13832| -86.7785|                       2|
|    20180000079|2018-01-01 06:26:00|   TN|   NASHVILLE|  BROADWAY & 7TH AVN|36.15941|-86.78091|                       2|
|    20180000088|2018-01

In [62]:
## перенос second_table в Postgres

hive_df = spark.table("etl_project_db.second_table")

last_month_data = hive_df.filter(
    (F.year('date') == 2018) & (F.month('date') == 1)  ## здесь устанавливаем тестовую дату (год+месяц)
)

## здесь можно написать любую агрегацию (в рамках разумного)
last_month_data = last_month_data.groupBy(
    F.year('date').alias('year'),
    F.month('date').alias('month'),
    'city'
).agg(
    F.count('number_of_motor_vehicles').alias('count_vehicles')
)

last_month_data.write \
    .format("jdbc") \
    .option("url", f"jdbc:postgresql://{node_address}:5432/mydatabase") \
    .option("dbtable", "number_of_motor_vehicles_per_month") \
    .option("user", "myuser") \
    .option("password", "mypassword") \
    .mode("overwrite") \
    .save()

In [86]:
## чтение third_table из Hive
third_table = spark.read.table("etl_project_db.third_table")

third_table.show(20)

AnalysisException: Table or view not found: etl_project_db.third_table;;
'UnresolvedRelation [etl_project_db, third_table]


In [64]:
## перенос third_table в Postgres

hive_df = spark.table("etl_project_db.third_table")

last_month_data = hive_df.filter(
    (F.year('date') == 2018) & (F.month('date') == 1)  ## здесь устанавливаем тестовую дату (год+месяц)
)

## здесь можно написать любую агрегацию (в рамках разумного)
last_month_data = last_month_data.withColumn('column', F.lit(1))

last_month_data = last_month_data.groupBy(
    F.year('date').alias('year'),
    F.month('date').alias('month'),
    'weather_description'
).agg(
    F.count('column').alias('count_weather_type')
)

last_month_data.write \
    .format("jdbc") \
    .option("url", f"jdbc:postgresql://{node_address}:5432/mydatabase") \
    .option("dbtable", "weather_type_count_per_month") \
    .option("user", "myuser") \
    .option("password", "mypassword") \
    .mode("overwrite") \
    .save()

In [65]:
## посмотрим, какие таблицы в PG получились

In [66]:
## создадим SparkSession для чтения таблиц из Postgres

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Read from PostgreSQL") \
    .config("spark.jars", "./postgresql-42.7.5.jar") \
    .getOrCreate()

In [85]:
## first_table
postgres_df = spark.read \
    .format("jdbc") \
    .option("url", f"jdbc:postgresql://{node_address}:5432/mydatabase") \
    .option("dbtable", "injuries_and_fatalities_per_month") \
    .option("user", "myuser") \
    .option("password", "mypassword") \
    .load()


postgres_df.show()

Py4JJavaError: An error occurred while calling o680.load.
: org.postgresql.util.PSQLException: ERROR: relation "injuries_and_fatalities_per_month" does not exist
  Position: 15
	at org.postgresql.core.v3.QueryExecutorImpl.receiveErrorResponse(QueryExecutorImpl.java:2733)
	at org.postgresql.core.v3.QueryExecutorImpl.processResults(QueryExecutorImpl.java:2420)
	at org.postgresql.core.v3.QueryExecutorImpl.execute(QueryExecutorImpl.java:372)
	at org.postgresql.jdbc.PgStatement.executeInternal(PgStatement.java:517)
	at org.postgresql.jdbc.PgStatement.execute(PgStatement.java:434)
	at org.postgresql.jdbc.PgPreparedStatement.executeWithFlags(PgPreparedStatement.java:194)
	at org.postgresql.jdbc.PgPreparedStatement.executeQuery(PgPreparedStatement.java:137)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCRDD$.resolveTable(JDBCRDD.scala:61)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCRelation$.getSchema(JDBCRelation.scala:226)
	at org.apache.spark.sql.execution.datasources.j

In [68]:
## second_table
postgres_df = spark.read \
    .format("jdbc") \
    .option("url", f"jdbc:postgresql://{node_address}:5432/mydatabase") \
    .option("dbtable", "number_of_motor_vehicles_per_month") \
    .option("user", "myuser") \
    .option("password", "mypassword") \
    .load()


postgres_df.show()

+----+-----+--------------+--------------+
|year|month|          city|count_vehicles|
+----+-----+--------------+--------------+
|2018|    1|       JOELTON|            12|
|2018|    1|     HERMITAGE|           112|
|2018|    1|  WHITES CREEK|            34|
|2018|    1|     NASHVILLE|          2265|
|2018|    1|     BRENTWOOD|            48|
|2018|    1|  MOUNT JULIET|             1|
|2018|    1|GOODLETTSVILLE|            13|
|2018|    1|   OLD HICKORY|            25|
|2018|    1|       ANTIOCH|           275|
|2018|    1|       MADISON|           199|
+----+-----+--------------+--------------+



In [69]:
## third_table
postgres_df = spark.read \
    .format("jdbc") \
    .option("url", f"jdbc:postgresql://{node_address}:5432/mydatabase") \
    .option("dbtable", "weather_type_count_per_month") \
    .option("user", "myuser") \
    .option("password", "mypassword") \
    .load()


postgres_df.show()

+----+-----+-------------------+------------------+
|year|month|weather_description|count_weather_type|
+----+-----+-------------------+------------------+
|2018|    1|                FOG|                14|
|2018|    1|               SNOW|               258|
|2018|    1|  OTHER (NARRATIVE)|                11|
|2018|    1|               null|               159|
|2018|    1|   SEVERE CROSSWIND|                 1|
|2018|    1|             CLOUDY|               305|
|2018|    1|               RAIN|               275|
|2018|    1|        SLEET, HAIL|                45|
|2018|    1|       BLOWING SNOW|                12|
|2018|    1|              CLEAR|              1876|
|2018|    1|            UNKNOWN|                28|
+----+-----+-------------------+------------------+



In [70]:
## если вдруг нужно дропнуть Hive и Postgres таблицы, выполните код в следующих ячейках

In [4]:
## удаление first_table из Hive

spark.sql("DROP TABLE IF EXISTS etl_project_db.first_table")

DataFrame[]

In [5]:
## удаление second_table из Hive

spark.sql("DROP TABLE IF EXISTS etl_project_db.second_table")

DataFrame[]

In [6]:
## удаление third_table из Hive

spark.sql("DROP TABLE IF EXISTS etl_project_db.third_table")

DataFrame[]

In [8]:
## создадим SparkSession для удаление Postgres таблиц

import psycopg2
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Delete Tables in Postgres") \
    .config("spark.jars", "./postgresql-42.7.5.jar") \
    .getOrCreate()

jdbc_url = f"jdbc:postgresql://{node_address}:5432/mydatabase"
properties = {"user": "myuser", "password": "mypassword", "driver": "org.postgresql.Driver"}

conn = psycopg2.connect(
    dbname="mydatabase",
    user="myuser",
    password="mypassword",
    host=node_address,
    port="5432"
)

cur = conn.cursor()

In [9]:
## удаление first_table из Postgres

cur.execute("DROP TABLE IF EXISTS injuries_and_fatalities_per_month")

In [10]:
## удаление second_table из Postgres

cur.execute("DROP TABLE IF EXISTS number_of_motor_vehicles_per_month")

In [11]:
## удаление third_table из Postgres

cur.execute("DROP TABLE IF EXISTS weather_type_count_per_month")

In [12]:
## закрываем курсор и соединение

cur.close()
conn.commit()
conn.close()